In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [66]:
%%writefile matmul.cu
__global__ void matmul(float* a, float*b, float*c, int M, int K, int N){

  int row = threadIdx.x + blockIdx.x * blockDim.x;
  int col = threadIdx.y + blockIdx.y * blockDim.y;
  if (row < M && col < N){
    c[row*N + col] = 0;
    for(int i = 0;i < K; i++){
      c[row*N + col] += a[row*K+i] * b[col+i*N];
    }
  }
}

Overwriting matmul.cu


In [67]:
%%writefile main.cu

#include "matmul.cu"
#include <stdio.h>
#include <chrono>

int main() {
  // declare in cpu
  int n = 4;
  int M = 500;
  int N = 500;
  int K = 500;
  float h_a[M][K];
  float h_b[K][N];
  float h_c[M][N];
  for(int i = 0; i < M ;i++){
    for(int j = 0;j < K; j++){
      h_a[i][j] = i*n+j+1;
    }
  }
  for(int i = 0; i < K ;i++){
    for(int j = 0;j < N; j++){
      if(i == j) h_b[i][j] = 1;
      else h_b[i][j] = 0;
    }
  }


  // declare for gpu
  float* d_a;
  cudaMalloc(&d_a, M*K * sizeof(float));
  float* d_b;
  cudaMalloc(&d_b, K*N * sizeof(float));
  float* d_c;
  cudaMalloc(&d_c, M*N * sizeof(float));

  // put cpu values in gpu
  auto start = std::chrono::high_resolution_clock::now();
  cudaMemcpy(d_a, h_a, M*K*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, h_b, K*N*sizeof(float), cudaMemcpyHostToDevice);
  auto end = std::chrono::high_resolution_clock::now();
  auto duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("copying 2 %d size values %lld\n", n, (long long)duration.count());


  // call kernel
  dim3 threadsPerBlock(16, 16);
  dim3 numBlocks((M + 15) / 16, (N + 15) / 16);
  // timing
  start = std::chrono::high_resolution_clock::now();
  matmul<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, M, K, N);
  cudaDeviceSynchronize();
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("gpu takes %lld time for %d size arrays\n", (long long)duration.count(), n);

  // put d_c in h_c
  start = std::chrono::high_resolution_clock::now();
  cudaMemcpy(h_c, d_c, M*N*sizeof(float), cudaMemcpyDeviceToHost);
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("duration to copy back %d values to cpu is %lld\n", n, (long long)duration.count());

  start = std::chrono::high_resolution_clock::now();
  for(int i = 0; i < M ;i++){
    for(int j = 0;j < N; j++){
      h_c[i][j] = 0;
      for(int k = 0;k < K; k++){
        h_c[i][j] += h_a[i][k] * h_b[k][j];
      }
    }

  }
  end = std::chrono::high_resolution_clock::now();
  duration = std::chrono::duration_cast<std::chrono::microseconds>(end - start);
  printf("duration for cpu to add %d values is %lld", n, (long long)duration.count());
}




Overwriting main.cu


In [68]:
!nvcc main.cu -o main && ./main

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
copying 2 4 size values 617
gpu takes 6206 time for 4 size arrays
duration to copy back 4 values to cpu is 298
duration for cpu to add 4 values is 653140